# Cross-Validation + MLflow
- **Experiment 1**: Kinect XY → Kinect Z
- **Experiment 2**: MediaPipe XY → MediaPipe Z

Both are logged to the **same** `mlruns/` folder so the running MLflow UI picks them up immediately.

## Cell 1 – Imports & device

In [1]:
import os, sys, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold
import matplotlib
matplotlib.use('Agg')          # headless – avoids display errors on servers
import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch

sys.path.append('../../MainProject/Assignment9')
from assignment9_functions import *

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

# ─────────────────────────────────────────────────────────────────────────────
# MLflow tracking URI
# This MUST point to the same folder your `mlflow ui` is watching.
# Default: mlflow ui reads from ./mlruns in whichever directory you ran it.
# Change MLRUNS_DIR if you launched the UI from a different directory.
# ─────────────────────────────────────────────────────────────────────────────
MLRUNS_DIR = os.path.abspath('./mlruns')
mlflow.set_tracking_uri(f'file://{MLRUNS_DIR}')
print(f'Tracking URI : {mlflow.get_tracking_uri()}')
print(f'MLflow UI    : http://127.0.0.1:5000')
print()
print('>>> If the UI was started from a DIFFERENT directory, update MLRUNS_DIR above')
print('    to match that directory, or restart the UI with:')
print(f'    mlflow ui --backend-store-uri {MLRUNS_DIR}')

Device: mps
Tracking URI : file:///Users/jakob/Documents/Universitet/Deep ML/DeepLearning_Schoolwork/JakobProject/notebooks/mlruns
MLflow UI    : http://127.0.0.1:5000

>>> If the UI was started from a DIFFERENT directory, update MLRUNS_DIR above
    to match that directory, or restart the UI with:
    mlflow ui --backend-store-uri /Users/jakob/Documents/Universitet/Deep ML/DeepLearning_Schoolwork/JakobProject/notebooks/mlruns


## Cell 2 – Paths & file lists

In [2]:
RANDOM_SEED = 112
K_FOLDS     = 5

KINECT_DIR = '../../MainProject/Assignment9/data/kinect_good_preprocessed'
MP_DIR     = '../../MainProject/Assignment10/data/csv_of_all_videos'

train_files_kinect, test_files_kinect = split_csvfiles(KINECT_DIR, RANDOM_SEED, 0.9, 0)
train_files_mp = [f.replace('kinect', 'mediapipe') for f in train_files_kinect]
test_files_mp  = [f.replace('kinect', 'mediapipe') for f in test_files_kinect]

train_pairs = list(zip(train_files_kinect, train_files_mp))
test_pairs  = list(zip(test_files_kinect,  test_files_mp))

print(f'Train videos : {len(train_pairs)}')
print(f'Test  videos : {len(test_pairs)}')

Train videos : 161
Test  videos : 18


## Cell 3 – Data helpers

In [3]:
def load_aligned_pair(kinect_file, mp_file):
    """Load one kinect + mediapipe CSV pair, aligned on FrameNo."""
    df_k = pd.read_csv(os.path.join(KINECT_DIR, kinect_file))
    df_m = pd.read_csv(os.path.join(MP_DIR,     mp_file))
    df_k.columns = df_k.columns.str.strip()
    frames = df_k['FrameNo'].values
    df_m   = df_m[df_m['FrameNo'].isin(frames)].reset_index(drop=True)
    df_k   = df_k.reset_index(drop=True)
    return df_k, df_m


def xy_z_tensors(df):
    """Split one dataframe: XY columns → X tensor, Z columns → Y tensor."""
    xy_cols = [c for c in df.columns if c.endswith('_x') or c.endswith('_y')]
    z_cols  = [c for c in df.columns if c.endswith('_z')]
    X = torch.tensor(df[xy_cols].values, dtype=torch.float32)
    Y = torch.tensor(df[z_cols].values,  dtype=torch.float32)
    return X, Y


def load_sensor_tensors(pairs, sensor):
    """
    Given a list of (kinect_file, mp_file) pairs, load and concatenate
    XY and Z data from the chosen sensor only.

    sensor = 'kinect'    ->  Kinect XY  -> Kinect Z
    sensor = 'mediapipe' ->  MP XY      -> MP Z
    """
    dfs = []
    for kf, mf in pairs:
        df_k, df_m = load_aligned_pair(kf, mf)
        dfs.append(df_k if sensor == 'kinect' else df_m)
    df_all = pd.concat(dfs, ignore_index=True)
    return xy_z_tensors(df_all)

## Cell 4 – Model

In [4]:
class DenseZPredictor(nn.Module):
    def __init__(self, input_size, output_size,
                 hidden_layers=(256, 128, 64), dropout=0.3):
        super().__init__()
        layers, prev = [], input_size
        for h in hidden_layers:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, output_size))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

## Cell 5 – Training & evaluation helpers

In [5]:
def train_fold(model, X_tr, Y_tr, X_val, Y_val, cfg):
    """
    Train one fold. Caller must have already opened an mlflow run.
    Logs one metric entry per epoch (visible as curves in the UI).
    Returns (best_model, history).
    """
    tr_dl  = DataLoader(TensorDataset(X_tr,  Y_tr),  batch_size=cfg['batch_size'], shuffle=True)
    val_dl = DataLoader(TensorDataset(X_val, Y_val), batch_size=cfg['batch_size'], shuffle=False)

    crit  = nn.MSELoss()
    opt   = optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg.get('wd', 0))
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=20)

    best_vl, best_state = float('inf'), None
    patience_ctr = 0
    history = {'train_loss': [], 'val_loss': [], 'train_mae': [], 'val_mae': []}

    for epoch in range(cfg['epochs']):
        model.train()
        tl_list, tm_list = [], []
        for bX, bY in tr_dl:
            bX, bY = bX.to(device), bY.to(device)
            opt.zero_grad()
            p = model(bX)
            loss = crit(p, bY)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl_list.append(loss.item())
            tm_list.append(torch.mean(torch.abs(p - bY)).item())

        model.eval()
        vl_list, vm_list = [], []
        with torch.no_grad():
            for bX, bY in val_dl:
                bX, bY = bX.to(device), bY.to(device)
                p = model(bX)
                vl_list.append(crit(p, bY).item())
                vm_list.append(torch.mean(torch.abs(p - bY)).item())

        tl = np.mean(tl_list); tm = np.mean(tm_list)
        vl = np.mean(vl_list); vm = np.mean(vm_list)
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_mae'].append(tm);  history['val_mae'].append(vm)

        sched.step(vl)

        # Log every epoch → smooth curves in MLflow UI
        mlflow.log_metrics({
            'train_loss': tl, 'train_mae': tm,
            'val_loss':   vl, 'val_mae':   vm,
            'lr': opt.param_groups[0]['lr']
        }, step=epoch)

        if vl < best_vl:
            best_vl = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if epoch % 50 == 0:
            print(f'    ep {epoch:3d}  tl={tl:.5f}  vl={vl:.5f}  vm={vm:.4f}')

        if cfg.get('patience') and patience_ctr >= cfg['patience']:
            print(f'    Early stop @ epoch {epoch}')
            break

    model.load_state_dict(best_state)
    return model, history


def eval_metrics(model, X, Y, prefix):
    """Compute MSE/MAE/R2/bias and log to the active mlflow run."""
    model.eval()
    with torch.no_grad():
        preds = model(X.to(device)).cpu().numpy()
    Yn = Y.numpy()
    m = {
        f'{prefix}_mse':  float(mean_squared_error(Yn.flatten(), preds.flatten())),
        f'{prefix}_mae':  float(mean_absolute_error(Yn.flatten(), preds.flatten())),
        f'{prefix}_r2':   float(r2_score(Yn.flatten(), preds.flatten())),
        f'{prefix}_bias': float(np.mean(preds.flatten() - Yn.flatten())),
    }
    mlflow.log_metrics(m)
    for k, v in m.items():
        print(f'    {k}: {v:.5f}')
    return m


def save_curve_artifact(history, fold_i, tag):
    """Save a training-curve PNG and attach it to the active mlflow run."""
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    a1.plot(history['train_loss'], label='train')
    a1.plot(history['val_loss'],   label='val')
    a1.set_title(f'{tag} fold {fold_i+1} - Loss'); a1.legend(); a1.grid(True)
    a2.plot(history['train_mae'],  label='train')
    a2.plot(history['val_mae'],    label='val')
    a2.set_title(f'{tag} fold {fold_i+1} - MAE');  a2.legend(); a2.grid(True)
    plt.tight_layout()
    path = f'/tmp/{tag}_fold{fold_i+1}.png'
    fig.savefig(path, dpi=120)
    mlflow.log_artifact(path)
    plt.close(fig)

## Cell 6 – Generic CV runner

In [6]:
def run_cv(experiment_name, sensor, train_pairs, test_pairs, cfg, k=K_FOLDS):
    """
    Full video-level K-Fold CV logged to MLflow.

    experiment_name : shown as a tab in the MLflow UI sidebar
    sensor          : 'kinect' | 'mediapipe'
    train_pairs     : list of (kinect_file, mp_file) used for CV
    test_pairs      : list of (kinect_file, mp_file) held out entirely
    cfg             : hyperparameter dict
    k               : number of folds

    MLflow structure
    ----------------
    Experiment: <experiment_name>
      Run: <sensor>_CV_parent          <- summary mean/std metrics here
        Run: fold_1  (nested)          <- per-epoch curves, model artifact
        Run: fold_2  (nested)
        ...
    """
    mlflow.set_experiment(experiment_name)

    # Hold-out test tensors (same sensor)
    X_test, Y_test = load_sensor_tensors(test_pairs, sensor=sensor)
    print(f'[{experiment_name}]  test frames={len(X_test)}  '
          f'in={X_test.shape[1]}  out={Y_test.shape[1]}')

    pairs_arr = np.array(train_pairs, dtype=object)
    kf        = KFold(n_splits=k, shuffle=True, random_state=RANDOM_SEED)
    all_metrics = []

    # ── PARENT RUN ────────────────────────────────────────────────────────────
    with mlflow.start_run(run_name=f'{sensor}_CV_parent') as parent:
        mlflow.log_params({
            'sensor':        sensor,
            'k_folds':       k,
            'hidden_layers': str(cfg['hidden_layers']),
            'dropout':       cfg['dropout'],
            'lr':            cfg['lr'],
            'batch_size':    cfg['batch_size'],
            'epochs':        cfg['epochs'],
            'weight_decay':  cfg.get('wd', 0),
            'patience':      cfg.get('patience', 'none'),
        })

        for fold_i, (tr_idx, val_idx) in enumerate(kf.split(pairs_arr)):
            print(f'\n  ── Fold {fold_i+1}/{k} ──────────────────────────────────')
            tr_fold  = pairs_arr[tr_idx].tolist()
            val_fold = pairs_arr[val_idx].tolist()

            X_tr,  Y_tr  = load_sensor_tensors(tr_fold,  sensor=sensor)
            X_val, Y_val = load_sensor_tensors(val_fold, sensor=sensor)
            print(f'  train={len(X_tr)} frames  |  val={len(X_val)} frames')

            in_sz  = X_tr.shape[1]
            out_sz = Y_tr.shape[1]

            # ── CHILD (NESTED) RUN ────────────────────────────────────────────
            with mlflow.start_run(run_name=f'fold_{fold_i+1}', nested=True):
                mlflow.log_params({
                    'fold':         fold_i + 1,
                    'train_frames': len(X_tr),
                    'val_frames':   len(X_val),
                    'input_size':   in_sz,
                    'output_size':  out_sz,
                })

                model = DenseZPredictor(
                    input_size=in_sz,
                    output_size=out_sz,
                    hidden_layers=cfg['hidden_layers'],
                    dropout=cfg['dropout'],
                ).to(device)
                mlflow.log_param('n_params',
                    sum(p.numel() for p in model.parameters()))

                model, history = train_fold(
                    model, X_tr, Y_tr, X_val, Y_val, cfg)

                print('  [val]')
                val_m  = eval_metrics(model, X_val,  Y_val,  prefix='val')
                print('  [test – held-out]')
                test_m = eval_metrics(model, X_test, Y_test, prefix='test')

                mlflow.pytorch.log_model(model, f'model_fold_{fold_i+1}')
                save_curve_artifact(history, fold_i, sensor)

                all_metrics.append({**val_m, **test_m})

        # ── SUMMARY → logged to parent run ────────────────────────────────────
        print(f'\n  ── {experiment_name}  SUMMARY ─────────────────────────')
        summary = {}
        for key in all_metrics[0]:
            vals = [m[key] for m in all_metrics]
            summary[key] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals))}
            mlflow.log_metric(f'{key}_mean', summary[key]['mean'])
            mlflow.log_metric(f'{key}_std',  summary[key]['std'])
            print(f'  {key:25s}  {summary[key]["mean"]:.5f} +/- {summary[key]["std"]:.5f}')

        print(f'\n  Parent run ID: {parent.info.run_id}')
    return all_metrics, summary

## Cell 7 – Hyperparameters

In [7]:
cfg = {
    'hidden_layers': [256, 128, 64],
    'dropout':       0.3,
    'lr':            0.001,
    'batch_size':    64,
    'epochs':        300,
    'wd':            1e-5,
    'patience':      30,
}

## Cell 8 – CV: Kinect XY → Kinect Z
Appears in MLflow UI as experiment **`Kinect_XY_to_Z`**

In [8]:
kinect_metrics, kinect_summary = run_cv(
    experiment_name = 'Kinect_XY_to_Z',
    sensor          = 'kinect',
    train_pairs     = train_pairs,
    test_pairs      = test_pairs,
    cfg             = cfg,
    k               = K_FOLDS,
)

/Users/jakob/Documents/Universitet/Deep ML/pytorch.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/04/22 21:11:27 INFO mlflow.tracking.fluent: Experiment with name 'Kinect_XY_to_Z' does not exist. Creating a new experiment.


[Kinect_XY_to_Z]  test frames=2312  in=26  out=13

  ── Fold 1/5 ──────────────────────────────────
  train=17462 frames  |  val=4231 frames
    ep   0  tl=0.00765  vl=0.00500  vm=0.0526
    ep  50  tl=0.00217  vl=0.00267  vm=0.0375
    ep 100  tl=0.00184  vl=0.00241  vm=0.0356
    ep 150  tl=0.00176  vl=0.00235  vm=0.0353
    Early stop @ epoch 164
  [val]
    val_mse: 0.00231
    val_mae: 0.03499
    val_r2: 0.87446
    val_bias: 0.00778
  [test – held-out]


2026/04/22 21:13:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/22 21:13:57 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


    test_mse: 0.00193
    test_mae: 0.03253
    test_r2: 0.90193
    test_bias: 0.00927

  ── Fold 2/5 ──────────────────────────────────
  train=17274 frames  |  val=4419 frames
    ep   0  tl=0.00752  vl=0.00393  vm=0.0458
    ep  50  tl=0.00223  vl=0.00219  vm=0.0330
    ep 100  tl=0.00196  vl=0.00194  vm=0.0315


2026/04/22 21:15:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/22 21:15:51 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


...

    Early stop @ epoch 116
  [val]
    val_mse: 0.00339
    val_mae: 0.03791
    val_r2: 0.82808
    val_bias: 0.00500
  [test – held-out]
    test_mse: 0.00199
    test_mae: 0.03334
    test_r2: 0.89929
    test_bias: 0.00960

  ── Fold 4/5 ──────────────────────────────────
  train=17522 frames  |  val=4171 frames
    ep   0  tl=0.00708  vl=0.00534  vm=0.0517
    ep  50  tl=0.00214  vl=0.00399  vm=0.0402


2026/04/22 21:18:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/22 21:18:35 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


    Early stop @ epoch 64
  [val]
    val_mse: 0.00361
    val_mae: 0.03765
    val_r2: 0.79721
    val_bias: 0.00208
  [test – held-out]
    test_mse: 0.00208
    test_mae: 0.03340
    test_r2: 0.89451
    test_bias: 0.01152

  ── Fold 5/5 ──────────────────────────────────
  train=17255 frames  |  val=4438 frames
    ep   0  tl=0.00828  vl=0.00426  vm=0.0494
    ep  50  tl=0.00226  vl=0.00283  vm=0.0376
    ep 100  tl=0.00201  vl=0.00232  vm=0.0342


2026/04/22 21:20:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/22 21:20:43 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


    Early stop @ epoch 132
  [val]
    val_mse: 0.00227
    val_mae: 0.03382
    val_r2: 0.87614
    val_bias: -0.00104
  [test – held-out]
    test_mse: 0.00188
    test_mae: 0.03226
    test_r2: 0.90456
    test_bias: 0.00744

  ── Kinect_XY_to_Z  SUMMARY ─────────────────────────
  val_mse                    0.00270 +/- 0.00068
  val_mae                    0.03513 +/- 0.00247
  val_r2                     0.85466 +/- 0.03657
  val_bias                   0.00163 +/- 0.00468
  test_mse                   0.00192 +/- 0.00012
  test_mae                   0.03241 +/- 0.00104
  test_r2                    0.90269 +/- 0.00621
  test_bias                  0.00863 +/- 0.00210

  Parent run ID: 083e0622acb547f097f19d285b258cd4


## Cell 9 – CV: MediaPipe XY → MediaPipe Z
Appears in MLflow UI as experiment **`MediaPipe_XY_to_Z`**

In [9]:
mp_metrics, mp_summary = run_cv(
    experiment_name = 'MediaPipe_XY_to_Z',
    sensor          = 'mediapipe',
    train_pairs     = train_pairs,
    test_pairs      = test_pairs,
    cfg             = cfg,
    k               = K_FOLDS,
)

2026/04/22 21:20:45 INFO mlflow.tracking.fluent: Experiment with name 'MediaPipe_XY_to_Z' does not exist. Creating a new experiment.


[MediaPipe_XY_to_Z]  test frames=2312  in=26  out=13

  ── Fold 1/5 ──────────────────────────────────
  train=17462 frames  |  val=4231 frames
    ep   0  tl=0.00732  vl=0.00376  vm=0.0445
    ep  50  tl=0.00273  vl=0.00205  vm=0.0317
    ep 100  tl=0.00223  vl=0.00142  vm=0.0257
    ep 150  tl=0.00216  vl=0.00146  vm=0.0263
    ep 200  tl=0.00200  vl=0.00129  vm=0.0244
    ep 250  tl=0.00196  vl=0.00129  vm=0.0241


2026/04/22 21:24:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/22 21:24:47 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


    Early stop @ epoch 274
  [val]
    val_mse: 0.00122
    val_mae: 0.02357
    val_r2: 0.94788
    val_bias: 0.00089
  [test – held-out]
    test_mse: 0.00141
    test_mae: 0.02547
    test_r2: 0.93747
    test_bias: -0.00095

  ── Fold 2/5 ──────────────────────────────────
  train=17274 frames  |  val=4419 frames
    ep   0  tl=0.00957  vl=0.00515  vm=0.0510
    ep  50  tl=0.00241  vl=0.00234  vm=0.0322
    ep 100  tl=0.00204  vl=0.00219  vm=0.0308
    ep 150  tl=0.00190  vl=0.00197  vm=0.0288
    ep 200  tl=0.00181  vl=0.00200  vm=0.0292
    ep 250  tl=0.00177  vl=0.00197  vm=0.0289


2026/04/22 21:29:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/22 21:29:17 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


...

  [val]
    val_mse: 0.00160
    val_mae: 0.02566
    val_r2: 0.92406
    val_bias: -0.00397
  [test – held-out]
    test_mse: 0.00131
    test_mae: 0.02455
    test_r2: 0.94176
    test_bias: -0.00089

  ── Fold 4/5 ──────────────────────────────────
  train=17522 frames  |  val=4171 frames
    ep   0  tl=0.00810  vl=0.00487  vm=0.0471
    ep  50  tl=0.00246  vl=0.00261  vm=0.0333
    ep 100  tl=0.00227  vl=0.00257  vm=0.0328
    ep 150  tl=0.00204  vl=0.00233  vm=0.0307
    ep 200  tl=0.00189  vl=0.00221  vm=0.0295
    ep 250  tl=0.00183  vl=0.00222  vm=0.0294


2026/04/22 21:38:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/22 21:38:33 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


    Early stop @ epoch 296
  [val]
    val_mse: 0.00219
    val_mae: 0.02918
    val_r2: 0.89780
    val_bias: -0.00055
  [test – held-out]
    test_mse: 0.00141
    test_mae: 0.02523
    test_r2: 0.93725
    test_bias: 0.00122

  ── Fold 5/5 ──────────────────────────────────
  train=17255 frames  |  val=4438 frames
    ep   0  tl=0.00887  vl=0.00375  vm=0.0426
    ep  50  tl=0.00265  vl=0.00193  vm=0.0308
    ep 100  tl=0.00253  vl=0.00181  vm=0.0296
    ep 150  tl=0.00241  vl=0.00156  vm=0.0271
    ep 200  tl=0.00219  vl=0.00148  vm=0.0264
    ep 250  tl=0.00205  vl=0.00118  vm=0.0239


2026/04/22 21:43:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/22 21:43:01 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


  [val]
    val_mse: 0.00110
    val_mae: 0.02307
    val_r2: 0.94917
    val_bias: -0.00034
  [test – held-out]
    test_mse: 0.00137
    test_mae: 0.02484
    test_r2: 0.93901
    test_bias: 0.00115

  ── MediaPipe_XY_to_Z  SUMMARY ─────────────────────────
  val_mse                    0.00160 +/- 0.00041
  val_mae                    0.02595 +/- 0.00244
  val_r2                     0.92778 +/- 0.01916
  val_bias                   -0.00026 +/- 0.00217
  test_mse                   0.00136 +/- 0.00004
  test_mae                   0.02491 +/- 0.00039
  test_r2                    0.93944 +/- 0.00197
  test_bias                  -0.00001 +/- 0.00098

  Parent run ID: 83a45562d0e744e2874f7f7be34c6d46


## Cell 10 – Side-by-side comparison table

In [10]:
rows = []
for metric in ['test_mse', 'test_mae', 'test_r2', 'test_bias']:
    k_vals  = [m[metric] for m in kinect_metrics]
    mp_vals = [m[metric] for m in mp_metrics]
    rows.append({
        'metric':          metric,
        'kinect_mean':     np.mean(k_vals),
        'kinect_std':      np.std(k_vals),
        'mediapipe_mean':  np.mean(mp_vals),
        'mediapipe_std':   np.std(mp_vals),
    })

df_cmp = pd.DataFrame(rows).set_index('metric')
print(df_cmp.round(5).to_string())

           kinect_mean  kinect_std  mediapipe_mean  mediapipe_std
metric                                                           
test_mse       0.00192     0.00012         0.00136        0.00004
test_mae       0.03241     0.00104         0.02491        0.00039
test_r2        0.90269     0.00621         0.93944        0.00197
test_bias      0.00863     0.00210        -0.00001        0.00098


---
## MLflow UI quick reference

After running cells 8 & 9 you will see **two experiments** in the left sidebar:

| Experiment | Parent run | Child runs |
|---|---|---|
| `Kinect_XY_to_Z` | `kinect_CV_parent` | `fold_1` … `fold_5` |
| `MediaPipe_XY_to_Z` | `mediapipe_CV_parent` | `fold_1` … `fold_5` |

Each **child run** has:
- Smooth epoch-by-epoch `train_loss`, `val_loss`, `train_mae`, `val_mae`, `lr` curves
- Final `val_*` and `test_*` metrics (MSE, MAE, R2, bias)
- Saved PyTorch model artifact
- Training-curve PNG artifact

Each **parent run** has `*_mean` / `*_std` summary metrics across all 5 folds.

### Still not seeing experiments in the UI?
The most common cause is a directory mismatch. Check:
```bash
# Where was the UI launched from?
# Option A: restart UI pointing at this notebook's mlruns folder:
cd /path/to/this/notebook
mlflow ui --backend-store-uri ./mlruns

# Option B: update MLRUNS_DIR in Cell 1 to match wherever mlflow ui is running from
```